# Day 2 notebook companion

Run in order with synthetic data. Mermaid diagrams render on the website. Setup and shared helpers are embedded; no checkout is required. Learner exercises report NOT ATTEMPTED until implemented. Reference checks are separate. Optional controls also have direct function calls.


In [ ]:
import importlib.metadata
import subprocess
import sys
for package, version in {"cryptography": "50.0.1", "matplotlib": "3.10.6", "ipywidgets": "8.1.7"}.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])
print("Dependencies ready. Restart if an older library was already imported, then run all cells.")


## Shared teaching helpers

Inspect this implementation. TLS uses real SSL objects over memory buffers and temporary test key files; no system trust changes or network listeners. The teaching KDF is not a standardized protocol key schedule.


In [ ]:
"""Day 2 teaching helpers. Real TLS over MemoryBIO; no sockets or trust-store changes."""
from datetime import datetime, timedelta, timezone
from pathlib import Path
import ssl
import tempfile
import hashlib
from cryptography import x509
from cryptography.x509.oid import NameOID, ExtendedKeyUsageOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives.kdf.hkdf import HKDF


def make_pki(expired=False):
    """Create an isolated root, intermediate, server, and client for this run."""
    now = datetime.now(timezone.utc)
    keys = {name: ec.generate_private_key(ec.SECP256R1())
            for name in ('root', 'intermediate', 'server', 'client')}
    names = {name: x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, 'Workshop ' + name)])
             for name in keys}
    certs = {}
    for name, issuer, ca, path_length, eku in [
        ('root', 'root', True, 1, None),
        ('intermediate', 'root', True, 0, None),
        ('server', 'intermediate', False, None, ExtendedKeyUsageOID.SERVER_AUTH),
        ('client', 'intermediate', False, None, ExtendedKeyUsageOID.CLIENT_AUTH),
    ]:
        end = now - timedelta(days=1) if expired and name == 'server' else now + timedelta(days=7)
        builder = (x509.CertificateBuilder().subject_name(names[name]).issuer_name(names[issuer])
                   .public_key(keys[name].public_key()).serial_number(x509.random_serial_number())
                   .not_valid_before(now - timedelta(days=2)).not_valid_after(end)
                   .add_extension(x509.BasicConstraints(ca=ca, path_length=path_length), critical=True)
                   .add_extension(x509.KeyUsage(digital_signature=True, content_commitment=False,
                       key_encipherment=False, data_encipherment=False, key_agreement=False,
                       key_cert_sign=ca, crl_sign=ca, encipher_only=False, decipher_only=False), critical=True)
                   .add_extension(x509.SubjectKeyIdentifier.from_public_key(keys[name].public_key()), False)
                   .add_extension(x509.AuthorityKeyIdentifier.from_issuer_public_key(keys[issuer].public_key()), False))
        if eku:
            builder = builder.add_extension(x509.ExtendedKeyUsage([eku]), False)
            builder = builder.add_extension(x509.SubjectAlternativeName([
                x509.DNSName('invoice.test' if name == 'server' else 'client.test')]), False)
        certs[name] = builder.sign(keys[issuer], hashes.SHA256())
    return keys, certs


def tls_trial(hostname='invoice.test', trust_root=True, expired=False,
              include_intermediate=True, mtls=False, send_client=True,
              client_wrong_eku=False):
    """Handshake and exchange application bytes. Failures raise ssl.SSLError.

    Private PEM files are disposable teaching keys in a temporary directory.
    Does not implement online revocation, networking, or authorization policy.
    """
    keys, certs = make_pki(expired)
    pem = lambda c: c.public_bytes(serialization.Encoding.PEM)
    with tempfile.TemporaryDirectory(prefix='workshop-pki-') as directory:
        base = Path(directory)
        for name in ('server', 'client'):
            selected = 'server' if name == 'client' and client_wrong_eku else name
            chain = pem(certs[selected])
            if include_intermediate or name == 'client':
                chain += pem(certs['intermediate'])
            (base / (name + '.pem')).write_bytes(chain)
            (base / (name + '.key')).write_bytes(keys[selected].private_bytes(
                serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8,
                serialization.NoEncryption()))
        server_context = ssl.SSLContext(ssl.PROTOCOL_TLS_SERVER)
        client_context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
        for context in (server_context, client_context):
            context.minimum_version = context.maximum_version = ssl.TLSVersion.TLSv1_3
        server_context.load_cert_chain(str(base / 'server.pem'), str(base / 'server.key'))
        if trust_root:
            client_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if mtls:
            server_context.verify_mode = ssl.CERT_REQUIRED
            server_context.load_verify_locations(cadata=pem(certs['root']).decode())
        if send_client:
            client_context.load_cert_chain(str(base / 'client.pem'), str(base / 'client.key'))
        ci, co, si, so = (ssl.MemoryBIO() for _ in range(4))
        client = client_context.wrap_bio(ci, co, server_hostname=hostname)
        server = server_context.wrap_bio(si, so, server_side=True)
        completed = [False, False]

        def transfer():
            for outgoing, incoming in ((co, si), (so, ci)):
                if outgoing.pending:
                    incoming.write(outgoing.read())

        for _ in range(100):
            for index, peer in enumerate((client, server)):
                if not completed[index]:
                    try:
                        peer.do_handshake()
                        completed[index] = True
                    except ssl.SSLWantReadError:
                        pass
            transfer()
            if all(completed):
                break
        else:
            raise RuntimeError('TLS handshake stalled')
        payload = b'synthetic confidential invoice'
        client.write(payload)
        transfer()
        assert server.read(4096) == payload
        return {'version': client.version(), 'cipher': client.cipher()[0],
                'client_authenticated': bool(server.getpeercert()),
                'application_bytes': len(payload)}


def expect_rejection(operation, exceptions):
    """Assert the negative case, without accepting a silent failure."""
    try:
        operation()
    except exceptions:
        return
    raise AssertionError('Expected rejection did not occur')


def derive_day2(secret, transcript, direction=b'alice-to-bob'):
    """Teaching KDF only, not a standardized TLS or hybrid key schedule."""
    return HKDF(algorithm=hashes.SHA256(), length=32, salt=None,
                info=b'workshop-day2:v1|' + hashlib.sha256(transcript).digest()
                + b'|' + direction).derive(secret)


# Session 9: Post-Quantum Key Establishment

**45 minutes taught · 75–100 minutes independently.** Instructor (see course website) · Notebook (see course website)

## Outcomes and setup

Explain KeyGen, Encapsulate and Decapsulate; distinguish a KEM ciphertext from an encrypted application message; derive an AEAD key from the established secret; and observe the difference between malformed input and implicit rejection. Use the Day 2 setup (see course website). This is real ML-KEM-768 through the pinned library, not a placeholder or a Kyber-labelled substitute.



## A KEM establishes secret material

Bob generates an encapsulation key (public) and decapsulation key (private). Alice uses Bob's public key to generate both a shared secret and a KEM ciphertext. She sends the ciphertext, not the secret. Bob decapsulates to recover the corresponding secret. Alice does not choose an invoice as input to encapsulation: use a KDF and AEAD for that invoice.

```mermaid
sequenceDiagram
    participant B as Bob — recipient
    participant A as Alice — sender
    B->>B: KeyGen creates encapsulation and decapsulation keys
    B->>A: Authenticated encapsulation key
    A->>A: Encapsulate produces secret and KEM ciphertext
    A->>B: KEM ciphertext
    B->>B: Decapsulate produces matching secret
    Note over A,B: KDF and AEAD protect application records
```

“Authenticated” is an assumption supplied by a surrounding protocol. The KEM does not identify Bob or Alice. Anyone holding Bob's public key can encapsulate to it. Replacing that public key recreates the identity problem from Session 4.

ML-KEM is standardized in FIPS 203 and based on module-lattice assumptions. “Post-quantum” means designed to resist known quantum attacks under its assumptions, not mathematically guaranteed safe forever. The parameter names are identifiers, not bit counts of symmetric security. We use ML-KEM-768 for one concrete teaching profile.


In [ ]:
from cryptography.hazmat.primitives.asymmetric.mlkem import MLKEM768PrivateKey, MLKEM768PublicKey
bob_kem = MLKEM768PrivateKey.generate()
public_bytes = bob_kem.public_key().public_bytes_raw()
alice_secret, kem_ciphertext = MLKEM768PublicKey.from_public_bytes(public_bytes).encapsulate()
bob_secret = bob_kem.decapsulate(kem_ciphertext)
assert alice_secret == bob_secret
assert (len(public_bytes), len(kem_ciphertext), len(alice_secret)) == (1184, 1088, 32)
print('PASS: ML-KEM-768 establishes the same secret; measured public key/ciphertext/secret = 1184/1088/32 bytes')


The library returns `(shared_secret, ciphertext)` in that order. Do not assume another library uses the same order. Its raw private serialization is a seed representation, not the expanded private-key size from every standards table. We measure public wire material and avoid exporting private keys.

## From secret to record protection

```mermaid
flowchart LR
    K["ML-KEM secret"] --> H["HKDF with version and transcript context"]
    H --> A["AES-256-GCM key"]
    M["Invoice bytes"] --> E["AEAD encrypt"]
    A --> E
    E --> R["Nonce, ciphertext and tag"]
```

The KEM ciphertext is part of key establishment; the AEAD ciphertext contains protected application data. Mixing up these two objects causes implementation errors. This teaching transcript has a fixed prefix and fixed-length public key/ciphertext fields. Production protocols must define their own exact serialization, roles, authentication, confirmation and replay rules.


In [ ]:
import secrets
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.exceptions import InvalidTag
transcript = b'ML-KEM-768:workshop:v1|' + public_bytes + kem_ciphertext
send_key = derive_day2(alice_secret, transcript)
receive_key = derive_day2(bob_secret, transcript)
nonce = secrets.token_bytes(12)
aad = b'invoice:v1|tenant=acme|id=7'
record = AESGCM(send_key).encrypt(nonce, b'confidential teaching invoice', aad)
assert AESGCM(receive_key).decrypt(nonce, record, aad) == b'confidential teaching invoice'
expect_rejection(lambda: AESGCM(receive_key).decrypt(nonce, record, aad + b'!'), InvalidTag)
print('PASS: KDF and AEAD round trip; altered context rejected')


## Implicit rejection is not successful authentication

A wrong-length KEM ciphertext is rejected as malformed. A correctly sized but invalid ciphertext can cause ML-KEM decapsulation to return a pseudorandom fallback secret rather than a visible validity flag. This implicit rejection is intentional. The receiver must not interpret “returned 32 bytes” as “Alice authenticated successfully.”


In [ ]:
expect_rejection(lambda: bob_kem.decapsulate(kem_ciphertext[:-1]), ValueError)
changed_ct = bytes([kem_ciphertext[0] ^ 1]) + kem_ciphertext[1:]
changed_secret = bob_kem.decapsulate(changed_ct)
assert len(changed_secret) == 32 and changed_secret != alice_secret
changed_key = derive_day2(changed_secret, transcript)
expect_rejection(lambda: AESGCM(changed_key).decrypt(nonce, record, aad), InvalidTag)
other_bob = MLKEM768PrivateKey.generate()
assert other_bob.decapsulate(kem_ciphertext) != alice_secret
print('PASS: truncated KEM input rejects; same-length corruption/wrong recipient cannot recover the record key')


We hold transcript bytes fixed in this negative case to isolate the secret change. Real peers also bind their received ciphertext into context. Do not expose detailed decapsulation validity oracles or release unauthenticated plaintext. Follow the protocol's failure behavior.

## What deployment adds

| Issue | Required reasoning |
| --- | --- |
| Identity | Authenticate the encapsulation key and peer roles through a specified protocol |
| Forward secrecy | A retained decapsulation key can recover secrets from recorded KEM ciphertexts; analyze ephemeral use and erasure |
| Size | Account for keys, ciphertexts, certificates, framing and transport limits |
| Performance | Measure key generation, encapsulation and decapsulation on actual targets, with distributions |
| Assurance | Standardized algorithm does not imply a FIPS-validated installation or whole-system compliance |

The public key and ciphertext total 2272 bytes in our isolated exchange, excluding identity credentials and protocol framing. A successful microbenchmark on a desktop says little about embedded memory, network fragmentation or overload resistance. A state actor may target weak key provisioning or endpoints even if ML-KEM's mathematics remains secure.

## Practice and answers

Predict what happens for a different recipient, modified AAD, and replay of the same intact record. Explain which result comes from KEM, AEAD or application policy.

<details><summary>Worked answers</summary>
<p>A different private key does not recover Alice's secret. Changed AAD causes AEAD failure. Replaying an intact record may still verify because this example has no replay state. KEM establishes material, AEAD checks a record under a key, and the application must enforce freshness. A static KEM key does not automatically provide forward secrecy.</p>
</details>

Continue to Lab 5 (see course website).

## Sources

Reviewed 22 September 2026: [FIPS 203 and errata notices](https://csrc.nist.gov/pubs/fips/203/final), [cryptography ML-KEM API](https://cryptography.io/en/stable/hazmat/primitives/asymmetric/mlkem/). Review current errata before delivery; no runtime fallback to a classical or fake KEM is permitted.


In [ ]:
print("PASS: completed session-09-ml-kem demonstrations; learner status is reported separately")
